# Notebook 3  --  End-to-End GCN on Cora

This notebook trains our from-scratch GCN on the Cora citation network and
reproduces the results from Kipf & Welling (2017).  It also uses t-SNE to
visualise how node embeddings evolve during training.

---

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.manifold import TSNE

from gnn.data.cora  import load_cora, CLASS_NAMES
from gnn.models.gcn import GCN
from gnn.train      import train, evaluate, plot_history

SEED = 42
np.random.seed(SEED)

## 1. Load and Inspect the Cora Dataset

Cora contains 2708 scientific papers (nodes), 5429 citation links (edges),
1433 bag-of-words features per paper, and 7 research-area class labels.

In [ ]:
A_hat, X, y, train_mask, val_mask, test_mask = load_cora(verbose=True)

print(f"\nA_hat shape : {A_hat.shape}")
print(f"X     shape : {X.shape}")
print(f"y     shape : {y.shape}")
print(f"Classes     : {CLASS_NAMES}")

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(9, 4))
counts = [(y == c).sum() for c in range(len(CLASS_NAMES))]
ax.bar(CLASS_NAMES, counts, color='steelblue')
ax.set_ylabel('Number of papers')
ax.set_title('Cora  --  class distribution')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Degree distribution
import scipy.sparse as sp

# Recover approximate degrees from A_hat diagonal structure
# (We'll reload the raw adjacency for analysis)
from gnn.data.cora import _get_cache_dir, _parse_content, _parse_cites
from pathlib import Path

cache = _get_cache_dir(None)
node_ids, _, _ = _parse_content(cache / 'cora/cora.content')
adj = _parse_cites(cache / 'cora/cora.cites', node_ids)
degrees = np.array(adj.sum(axis=1)).flatten()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(degrees, bins=range(0, int(degrees.max())+2), color='coral', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Node degree')
ax.set_ylabel('Count')
ax.set_title(f'Degree distribution  (mean={degrees.mean():.1f}, max={int(degrees.max())})')
ax.set_xlim(-0.5, 20)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Build and Inspect the Model

In [ ]:
model = GCN(
    in_features=X.shape[1],   # 1433
    hidden_dim=64,
    n_classes=len(CLASS_NAMES),  # 7
    dropout=0.5,
    seed=SEED,
)
print(model)

## 3. Embeddings Before Training (Random Weights)

Before training, node embeddings are random.  We use t-SNE to project the
64-dimensional hidden layer to 2D.

In [ ]:
def get_embeddings(model, A_hat, X):
    """Extract hidden-layer (layer 1 output) embeddings."""
    model.eval()
    from gnn.autograd.tensor import Tensor
    H = model.layer1(A_hat, Tensor(X))
    return H.data

def plot_tsne(embeddings, labels, title, ax):
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
    emb2d = tsne.fit_transform(embeddings)
    colors = cm.tab10(np.linspace(0, 1, len(CLASS_NAMES)))
    for c, color in enumerate(colors):
        mask = labels == c
        ax.scatter(emb2d[mask, 0], emb2d[mask, 1],
                   c=[color], label=CLASS_NAMES[c], s=4, alpha=0.7)
    ax.set_title(title)
    ax.axis('off')

print("Computing t-SNE on random (untrained) embeddings ...")
emb_before = get_embeddings(model, A_hat, X)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
plot_tsne(emb_before, y, 'Before training (random weights)', ax)
ax.legend(loc='upper right', markerscale=3, fontsize=7)
plt.tight_layout()
plt.show()
print("Classes are mixed  --  the network has learned nothing yet.")

## 4. Training

In [ ]:
forward = lambda mask: model(A_hat, X, y, mask)
history = train(
    model, forward, y, train_mask, val_mask,
    epochs=200,
    lr=0.01,
    weight_decay=5e-4,
    patience=20,
    grad_clip=5.0,
    keep_best=True,
    verbose=True,
    log_every=25,
)

In [ ]:
plot_history(history)

## 5. Test Set Evaluation

In [ ]:
results = evaluate(model, forward, y, test_mask)
print(f"Test loss     : {results['test_loss']:.4f}")
print(f"Test accuracy : {results['test_acc']*100:.2f}%")
print()
print("Benchmark comparison:")
print(f"  Kipf & Welling (2017) paper: ~81.5%")
print(f"  Our implementation          :  {results['test_acc']*100:.1f}%")

## 6. Embeddings After Training

After training, the graph-convolved features should cluster by class.

In [ ]:
print("Computing t-SNE on trained embeddings ...")
emb_after = get_embeddings(model, A_hat, X)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_tsne(emb_before, y, 'Before training', axes[0])
plot_tsne(emb_after,  y, 'After training',  axes[1])
axes[1].legend(loc='upper right', markerscale=3, fontsize=7)
plt.suptitle('t-SNE of GCN hidden-layer embeddings (Cora)', fontsize=13)
plt.tight_layout()
plt.show()
print("Trained embeddings form tight, well-separated clusters.")

## 7. Per-Class Accuracy Breakdown

In [ ]:
model.eval()
from gnn.autograd.tensor import Tensor
log_probs, _ = model(A_hat, X)
preds = log_probs.data.argmax(axis=1)

per_class_acc = []
for c in range(len(CLASS_NAMES)):
    class_test_mask = test_mask & (y == c)
    if class_test_mask.sum() > 0:
        acc = (preds[class_test_mask] == y[class_test_mask]).mean()
    else:
        acc = 0.0
    per_class_acc.append(acc)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(CLASS_NAMES, [a*100 for a in per_class_acc], color='steelblue')
ax.axhline(results['test_acc']*100, color='red', linestyle='--', label=f"Overall: {results['test_acc']*100:.1f}%")
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-class accuracy on test set')
ax.tick_params(axis='x', rotation=30)
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc*100:.0f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Hyperparameter Ablation

How does hidden dimension affect performance?

In [ ]:
results_table = []
for hidden in [16, 32, 64, 128]:
    m = GCN(in_features=X.shape[1], hidden_dim=hidden, n_classes=7,
            dropout=0.5, seed=SEED)
    fwd = lambda mask, m=m: m(A_hat, X, y, mask)
    train(m, fwd, y, train_mask, val_mask,
          epochs=200, lr=0.01, weight_decay=5e-4, verbose=False)
    r = evaluate(m, fwd, y, test_mask)
    results_table.append((hidden, r['test_acc']))
    print(f"  hidden={hidden:3d}  test_acc={r['test_acc']*100:.2f}%")

hidden_dims, accs = zip(*results_table)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(hidden_dims, [a*100 for a in accs], 'o-', color='steelblue')
ax.set_xlabel('Hidden dimension')
ax.set_ylabel('Test accuracy (%)')
ax.set_title('Effect of hidden dimension')
ax.set_xticks(hidden_dims)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. What the GCN Learned

The GCN learns to classify papers by combining:
1. **Local graph structure**  --  who cites whom
2. **Node features**  --  bag-of-words content

Crucially, with only **140 labelled nodes** (out of 2708), the network
propagates label information through the citation graph to classify
the remaining nodes.  This is **transductive semi-supervised learning**.

The two-layer architecture means each node aggregates features from its
**2-hop neighbourhood**  --  papers that its papers cite, and papers that
cite those papers.

---

## 10. MLP Baseline -- What Happens Without the Graph?

An MLP ignores all citation links and classifies each paper from its
bag-of-words features alone.  The gap versus GCN directly measures
how much the graph topology contributes.

In [ ]:
from gnn.models.mlp import MLP

mlp = MLP(in_features=X.shape[1], hidden_dim=64, n_classes=len(CLASS_NAMES),
          dropout=0.5, seed=SEED)
print(mlp)

mlp_forward = lambda mask: mlp(X, y, mask)
mlp_history = train(
    mlp, mlp_forward, y, train_mask, val_mask,
    epochs=200, lr=0.01, weight_decay=5e-4,
    patience=20, grad_clip=5.0, keep_best=True,
    log_every=25,
)

mlp_metrics = evaluate(mlp, mlp_forward, y, test_mask)
mlp_acc = mlp_metrics['test_acc']
print(f"\nMLP  test accuracy: {mlp_acc*100:.2f}%")
print(f"GCN  test accuracy: {results['test_acc']*100:.2f}%")
print(f"Graph topology adds: {(results['test_acc'] - mlp_acc)*100:.1f} percentage points")

---

## 11. Graph Attention Network (GAT)

GAT replaces the fixed degree-based aggregation weights of GCN with
**learned, edge-specific attention scores**.

For each edge $(i, j)$, the attention weight is:

$$\alpha_{ij} = \text{softmax}_j\!\left(\text{LeakyReLU}\!\left(\mathbf{a}_l^\top (HW)_i + \mathbf{a}_r^\top (HW)_j\right)\right)$$

Nodes with more relevant neighbours attend to them more strongly.
The vectors $\mathbf{a}_l$ and $\mathbf{a}_r$ are learned alongside $W$.

In [ ]:
from gnn.models.gat import GAT
from gnn.train import train, evaluate

# GAT needs the raw (un-normalised) binary adjacency
from gnn.data.cora import _get_cache_dir, _parse_content, _parse_cites
cache = _get_cache_dir(None)
node_ids, _, _ = _parse_content(cache / 'cora/cora.content')
adj = _parse_cites(cache / 'cora/cora.cites', node_ids)
A_raw = adj.toarray().astype(float)

# Paper-faithful 8 heads in layer 1, 1 averaged head in layer 2.
# GATLayer is sparse end-to-end now (see gat-fix.md), so this no longer
# blows up memory the way it did when each head materialised an (N x N)
# attention matrix.
gat = GAT(in_features=X.shape[1], hidden_dim=64, n_classes=len(CLASS_NAMES),
          dropout=0.6, n_heads=8, seed=SEED)
print(gat)

gat_forward = lambda mask: gat(A_raw, X, y, mask)
gat_history = train(
    gat, gat_forward, y, train_mask, val_mask,
    epochs=100, lr=0.005, weight_decay=5e-4,
    patience=20, grad_clip=5.0, keep_best=True,
    log_every=25,
)

gat_metrics = evaluate(gat, gat_forward, y, test_mask)
gat_acc = gat_metrics['test_acc']
print(f"\nGAT test accuracy: {gat_acc*100:.2f}%  (best epoch {gat_history.best_epoch})")

### 11a. Visualising Attention Weights

After training, `layer1.last_alpha` contains the learned attention matrix.
We can inspect which neighbours a given node pays most attention to.

In [ ]:
from gnn.autograd.tensor import Tensor

# Trigger a forward pass to populate last_alpha
gat.eval()
gat.layer1(A_raw, Tensor(X))
# `last_alpha` is now a property that lazily reconstructs the dense
# (N, N) attention matrix from the cached edge values; the forward pass
# itself never built it.
alpha = gat.layer1.last_alpha   # (N x N) attention weights

# Pick 6 nodes spread across classes and show their top-5 neighbours by attention
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

sample_nodes = [np.where((y == c) & test_mask)[0][0] for c in range(6)]

for ax, node in zip(axes.flat, sample_nodes):
    nbrs = np.where(A_raw[node] > 0)[0]
    attn = alpha[node, nbrs]
    order = np.argsort(attn)[::-1][:8]
    top_nbrs = nbrs[order]
    top_attn = attn[order]

    colors = ['#E53935' if y[n] == y[node] else '#1E88E5' for n in top_nbrs]
    ax.barh(range(len(top_nbrs)), top_attn[::-1], color=colors[::-1])
    ax.set_yticks(range(len(top_nbrs)))
    ax.set_yticklabels([f"node {n}\n({CLASS_NAMES[y[n]]})" for n in top_nbrs[::-1]], fontsize=7)
    ax.set_xlabel('Attention weight')
    ax.set_title(f'Node {node}  [{CLASS_NAMES[y[node]]}]', fontsize=9)
    ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color='#E53935', label='Same class'),
                    Patch(color='#1E88E5', label='Different class')],
           loc='lower center', ncol=2, fontsize=9, framealpha=0.8)
plt.suptitle('Top-8 neighbours by attention weight (GAT layer 1)', fontsize=12)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

---

## 12. Model Comparison

In [ ]:
model_results = [
    ("MLP\n(no graph)",  mlp_acc,            "#DD8452"),
    ("GCN\n(Kipf 2017)", results['test_acc'], "#4C72B0"),
    ("GAT\n(Velickovic 2018)", gat_acc,       "#55A868"),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
names, accs, colors = zip(*model_results)
bars = axes[0].bar(names, [a*100 for a in accs], color=colors, width=0.5)
axes[0].set_ylabel("Test accuracy (%)")
axes[0].set_title("Cora test accuracy by model")
axes[0].set_ylim(50, 90)
axes[0].grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{acc*100:.1f}%", ha='center', va='bottom', fontweight='bold')

# Loss curves
axes[1].plot(mlp_history.val_loss, label='MLP',  color="#DD8452", linestyle="--")
axes[1].plot(history.val_loss,     label='GCN',  color="#4C72B0")
axes[1].plot(gat_history.val_loss, label='GAT',  color="#55A868", linestyle="-.")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation NLL loss")
axes[1].set_title("Validation loss curves")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("MLP vs GCN vs GAT on Cora", fontsize=13)
plt.tight_layout()
plt.show()

print("\nSummary")
print("-" * 42)
print(f"{'Model':<20} {'Test acc':>10}  {'Params':>8}")
print("-" * 42)
print(f"{'MLP':<20} {mlp_acc*100:>9.2f}%  {mlp.n_parameters():>8,}")
print(f"{'GCN':<20} {results['test_acc']*100:>9.2f}%  {model.n_parameters():>8,}")
print(f"{'GAT':<20} {gat_acc*100:>9.2f}%  {gat.n_parameters():>8,}")
print("-" * 42)
print(f"Graph topology adds {(results['test_acc'] - mlp_acc)*100:.1f}pp over MLP (GCN)")

---

## Summary

| Component | What it does |
|-----------|-------------|
| `Tensor` autograd | Differentiates through matrix ops without PyTorch |
| Cora loader | Downloads, parses, and normalises the citation graph |
| `GCNLayer` | Fixed degree-normalised neighbourhood aggregation |
| `GATLayer` | Learned per-edge attention weights via $\mathbf{a}_l$, $\mathbf{a}_r$ |
| `GCN` | Two GCN layers + dropout -> log-softmax predictions |
| `GAT` | Two GAT layers + dropout -> log-softmax predictions |
| `MLP` | Two linear layers, no graph -- baseline without topology |
| `Adam` | Adaptive optimiser with bias-corrected moments |
| t-SNE | Confirms embeddings cluster by research area after training |